In [42]:
# Imports
import pprint
from tokenize import Double

from jsonschema.validators import Draft7Validator
import json

from sqlalchemy.testing.schema import mapped_column
from sqlalchemy.util.preloaded import sql_schema

from src.mosdex.mosdex_db import MosdexBase

In [43]:
"""
Open a Mosdex file and validate it against the schema
"""

MOSDEX_SCHEMA_FILE = "../data/MOSDEXSchemaV2-1.json"
MOSDEX_FILE = "../data/sailco_2-1.json"

with open(MOSDEX_SCHEMA_FILE, "r") as f:
    schema = json.load(f)
validator = Draft7Validator(schema)

with open(MOSDEX_FILE, "r") as f:
    mosdex = json.load(f)

if not validator.is_valid(mosdex):
    print(f"File {MOSDEX_FILE} is not a valid Mosdex file.")
    pp = pprint.PrettyPrinter(indent=4)
    for error in sorted(validator.iter_errors(mosdex), key=str):
        print()
        pp.pprint(error.message)
else:
    print(f"File {MOSDEX_FILE} is a valid instance of schema {MOSDEX_SCHEMA_FILE}.")


File ../data/sailco_2-1.json is a valid instance of schema ../data/MOSDEXSchemaV2-1.json.


In [44]:
"""
Print some features of the MOSDEX_FILE
"""

print(f"The syntax of {MOSDEX_FILE} is {mosdex['SYNTAX']}")
for module in mosdex['MODULES']:
    print(f"\nModule {module['NAME']} is a {module['KIND']}")
    

The syntax of ../data/sailco_2-1.json is MOSDEX/MOSDEX v2-0/MOSDEXSchemaV2-0.json

Module sailco is a MODEL


In [45]:
# Get reference to a module that has KIND == MODEL
model = {}
for module in mosdex['MODULES']:
    if module['KIND'] == 'MODEL':
        model = module
        break

print(f"Got handle to MODEL: {model['NAME']}")
print(f"The sections of the model are {list(model.keys())}")
print(f"\tNAME: {model['NAME']}")
print(f"\tCLASS: {model['CLASS']}")
print(f"\tKIND: {model['KIND']}")
print(f"\tTABLES: there are {len(model['TABLES'])} tables:")
for table in model['TABLES']:
    print(f"\t\t{table['NAME']:10s} \t class/kind: {table['CLASS']}/{table['KIND']}")

Got handle to MODEL: sailco
The sections of the model are ['NAME', 'CLASS', 'KIND', 'HEADING', 'TABLES']
	NAME: sailco
	CLASS: MODULE
	KIND: MODEL
	TABLES: there are 11 tables:
		demands    	 class/kind: DATA/INPUT
		parameters 	 class/kind: DATA/INPUT
		regular    	 class/kind: VARIABLE/CONTINUOUS
		extra      	 class/kind: VARIABLE/CONTINUOUS
		inventory  	 class/kind: VARIABLE/CONTINUOUS
		cost       	 class/kind: VARIABLE/CONTINUOUS
		totalCost  	 class/kind: VARIABLE/CONTINUOUS
		ctCapacity 	 class/kind: CONSTRAINT/LINEAR
		ctBoat     	 class/kind: CONSTRAINT/LINEAR
		ctCost     	 class/kind: CONSTRAINT/LINEAR
		production 	 class/kind: DATA/OUTPUT


In [59]:
"""
Initialize the database engine
"""
from sqlalchemy import create_engine, Integer, String, Double, ForeignKey

engine = create_engine('sqlite:///:memory:')

from sqlalchemy.orm import declarative_base, mapped_column

Base = declarative_base()

# Declare each table
for table_ in model['TABLES']:
    table_name = table_['NAME']
    table_class = table_['CLASS']
    table_kind = table_['KIND']
    table_schema = table_['SCHEMA']
    
    # Dictionary for database info
    db_table: dict = {} 
    
    # Class name for rows
    db_table['class_name'] = table_['CLASS'] + '_' + model['NAME'] + '_' + table_['NAME']
    
    # Load the table_attr dictionary
    db_table['attr'] = { '__tablename__': table_name,
                      'id': mapped_column(Integer, primary_key=True)}
    
    # Apply the SCHEMA to the table columns
    for name, kind in zip(table_schema['NAME'], table_schema['KIND']):
        
        # Column type
        if kind == 'INTEGER':
            type_col = Integer
        elif kind == 'DOUBLE' or kind == 'DOUBLE_FUNCTION':
            type_col = Double
        elif kind == 'STRING':
            type_col = String
        else:
            print(f"Error, type {kind} is not supported")
            break
        
        # Add the column to the table_attr dictionary
        # - set primary_key flag 
        # - set Foreign Key relationship 
        if 'KEYS' in table_schema and name in table_schema['KEYS']:
            # Primary Key
            db_table['attr'][name] = mapped_column(type_col, primary_key=True)
        if 'FOREIGN_KEYS' in table_schema and name in table_schema['FOREIGN_KEYS']:
            # Foreign Key
            f_key = table_schema['FOREIGN_KEYS'][name]
            db_table['attr'][name] = mapped_column(type_col, ForeignKey(f_key))
        else:
            db_table['attr'][name] = mapped_column(type_col)
    
    # Declarative instantiation of the table     
    db_table['instance'] = type(db_table['class_name'], (Base,), db_table['attr'])
    
    # Add the db_table dictionary to the TABLE dict
    table_['db_table'] = db_table
    
    
Base.metadata.create_all(engine)
        
print(f"Database tables created")
for table in Base.metadata.tables.keys():
    print(f"\t{table}")
    print(f"\t\t{Base.metadata.tables[table].columns.keys()}")
    

Database tables created
	demands
		['id', 'period', 'ancestor', 'demand']
	parameters
		['id', 'regularCost', 'extraCost', 'capacity', 'initialInventory', 'inventoryCost']
	regular
		['id', 'Name', 'Period', 'ColumnName', 'LowerBound', 'UpperBound', 'Value']
	extra
		['id', 'Name', 'period', 'Column', 'LowerBound', 'UpperBound', 'value']
	inventory
		['id', 'Name', 'period', 'Column', 'LowerBound', 'UpperBound', 'value']
	cost
		['id', 'Name', 'period', 'Column', 'LowerBound', 'UpperBound', 'value']
	totalCost
		['id', 'Name', 'period', 'Column', 'LowerBound', 'UpperBound', 'value']
	ctCapacity
		['id', 'Name', 'period', 'Row', 'Sense', 'RHS', 'dual']
	ctBoat
		['id', 'Name', 'period', 'Row', 'Sense', 'RHS', 'dual']
	ctCost
		['id', 'Name', 'period', 'Row', 'Constant', 'Sense', 'Value']
	production
		['id', 'period', 'regular', 'extra', 'inventory', 'marginalCapacityValue']


In [60]:
from sqlalchemy import text
from sqlalchemy.orm import Session

"""
Load the INPUT data
"""
import pandas as pd
import numpy as np

for table_ in model['TABLES']:
    if table_['KIND'] == 'INPUT':
        if "INSTANCE" in table_ and table_['CLASS'] == "DATA":
            # Get column names 
            col_names = table_['SCHEMA']['NAME']
            
            # Create a dataframe from the INSTANCE arrays
            data_df = pd.DataFrame(np.vstack(table_['INSTANCE']), columns=col_names)
            print(data_df.head())
    
            # Push the dataframe to the table
            with Session(engine) as session, session.begin():
                data_df.to_sql(name=table_['NAME'], con=session.connection(),
                                       if_exists='append', index=False)
                session.flush()
                stmt = "select * from " + table_['NAME']
                print(session.execute(text(stmt)).fetchall())
        else:
            # Load in another way
            pass
    else:
        # Not INPUT 
        pass


   period  ancestor  demand
0     1.0       0.0    40.0
1     2.0       1.0    60.0
2     3.0       2.0    75.0
3     4.0       3.0    25.0
[(1, 1, 0, 40.0), (2, 2, 1, 60.0), (3, 3, 2, 75.0), (4, 4, 3, 25.0)]
   regularCost  extraCost  capacity  initialInventory  inventoryCost
0        400.0      450.0      40.0              10.0           20.0
[(1, 400.0, 450.0, 40.0, 10.0, 20.0)]


In [80]:
from sqlalchemy import Table

"""
Process VARIABLES
"""

# VARIABLE regular
"""
"QUERY": {
            "SELECT": [
              "'regular', ",
              "period, ",
              "CONCAT('regular','_', period), ",
              "0.0, ",
              "'Infinity', ",
              "NULL "
            ],
            "FROM": [
              "demands"
            ]
"""

# Get the demands table from the engine
demands = Table('demands', Base.metadata, autoload_with=engine)
regular = Table('regular', Base.metadata, autoload_with=engine)

# Test the SQL string
with Session(engine) as session, session.begin():
    stmt = text("SELECT "
                "'regular' AS Name, "
                "period AS Period, "
                "CONCAT('regular','_', period) AS ColumnName, "
                "0.0 AS LowerBound, "
                "'Infinity' AS UpperBound, "
                "NULL AS Value "
                "FROM demands")
    result = session.execute(stmt)
    for row in result:
        print(row)

col_types = {'Name': str, 'Period': np.int32, 'ColumnName': str, 'LowerBound': np.float64, 'UpperBound': np.float64, 'Value': np.float64}
df_regular = pd.read_sql(stmt, engine, dtype=col_types)
df_regular.head()

('regular', 1, 'regular_1', 0.0, 'Infinity', None)
('regular', 2, 'regular_2', 0.0, 'Infinity', None)
('regular', 3, 'regular_3', 0.0, 'Infinity', None)
('regular', 4, 'regular_4', 0.0, 'Infinity', None)


,Name,Period,ColumnName,LowerBound,UpperBound,Value
0,regular,1,regular_1,0.0,inf,NaN
1,regular,2,regular_2,0.0,inf,NaN
2,regular,3,regular_3,0.0,inf,NaN
3,regular,4,regular_4,0.0,inf,NaN
